In [6]:
import requests
import urllib
import os
import re
from pathlib import Path
import pandas as pd
import soundfile as sf

In [2]:
# https://github.com/SecFathy/YandexDown

class YandexDiskDownloader:
    def __init__(self, link, download_location):
        self.link = link
        self.download_location = download_location

    def download(self):
        url = f"https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key={self.link}"
        response = requests.get(url)
        download_url = response.json()["href"]
        file_name = urllib.parse.unquote(download_url.split("filename=")[1].split("&")[0])
        save_path = os.path.join(self.download_location, file_name)
        with open(save_path, "wb") as file:
            download_response = requests.get(download_url, stream=True)
            for chunk in download_response.iter_content(chunk_size=1024):
                if chunk:
                    file.write(chunk)
                    file.flush()
        print("Download complete.")
    
downloader = YandexDiskDownloader("https://disk.yandex.ru/d/h1SEmkqBro8Ffg", ".")
downloader.download()

Download complete.


In [3]:
!rm -rf good bad
!tar -xzf data.tar.gz

В датасете присутствуют дубликаты, избавимся от них

In [4]:
def delete_duplicates(directory_path):
    print(f"Directory {directory_path}. Start search duplicates")
    path = Path(directory_path)
    files_to_delete = []
    for file_path in path.iterdir():
        if file_path.is_file():
            filename = file_path.name
            if "(1)" in filename:
                parts = filename.rsplit("(1)", 1)
                if len(parts) == 2:
                    original_name = parts[0] + parts[1]
                    original_file_path = path / original_name
                    is_same_size = os.path.getsize(file_path) == os.path.getsize(original_file_path)
                    if original_file_path.is_file() and is_same_size:
                        files_to_delete.append(file_path)
    if files_to_delete:
        for f in files_to_delete:
            os.remove(f)
            print(f"Deleted: {f.name.encode('iso-8859-5').decode('utf-8')}")        
        print(f"Directory {directory_path}. Deletion finished.")
    else:
        print(f"Directory {directory_path}. No duplicates found.")

delete_duplicates('good')
delete_duplicates('bad')

Directory good. Start search duplicates


Deleted: eu.10e21015-42c9-4ef8-a8c2-66d4057c5423__чстрш(1).wav
Deleted: eu.1131c949-c891-44b0-b35f-78fa5be195ae__ст(1).wav
Deleted: eu.2b04a3f2-5397-4eea-a93c-ac145a0cbe78__рл(1).wav
Deleted: eu.3d5af675-092a-4f3e-9265-bdc70bcb2c17__чстр(1).wav
Deleted: eu.468b4fa0-a72e-46e2-800c-b781dd918dc7__стрш(1).wav
Deleted: eu.47035742-6d06-495b-811f-612d58d6d0e5__срл(1).wav
Deleted: eu.6056a741-3040-4376-9ad7-f59f83afbace__трлш(1).wav
Deleted: eu.67fa1c0b-518e-4155-81a7-0cdcd9432887__сл(1).wav
Deleted: eu.7c595ed0-3958-43f3-a29d-3700a9e5c750__чсрлц(1).wav
Deleted: eu.a2480375-3559-4aab-a6da-86a76935afa8__стрш(1).wav
Directory good. Deletion finished.
Directory bad. Start search duplicates
Directory bad. No duplicates found.


Сохраним в датафрейм информацию о файлах (битрейт, длительность и теги, которые содержатся в именах файлов) После чего файлы переименовываем, чтобы избавится от кириллицы.

In [7]:
def read_and_rename_files(dir_name):
    result = []
    folder_path = Path(dir_name)
    for file_path in folder_path.rglob('*.wav'):
        if file_path.is_file():
            filename = file_path.name.encode('iso-8859-5').decode('utf-8')

            new_filename = re.split(r'[._]+', file_path.name)[1] + '.wav'            
            new_file_path = file_path.with_name(new_filename)

            f = sf.SoundFile(file_path)
            result.append({
                "filename": dir_name + '/' + new_filename,
                "duration": f.frames / f.samplerate,
                "samplerate": f.samplerate,
                "tags": filename.split('__', 1)[1].split('.', 1)[0],
            })
                     
            try:
                file_path.rename(new_file_path)
            except OSError as e:
                print(f"Error renaming file {file_path.name}: {e}")
    return result

good_files = read_and_rename_files('good')
bad_files = read_and_rename_files('bad')

In [8]:
df_good = pd.DataFrame(good_files)
df_good['target'] = 1
df_bad = pd.DataFrame(bad_files)
df_bad['target'] = 0
df = pd.concat([df_good, df_bad], ignore_index=True)
df

,filename,duration,samplerate,tags,target
0,good/00f3da84-570e-40f5-9949-31cd811ff6ca.wav,12.415625,16000,чстрлшщ,1
1,good/01062fc6-fbb8-4544-a0f5-2830899a91d0.wav,6.480000,16000,чстр,1
2,good/01611791-7a64-42c6-82f4-1f8d46603999.wav,11.940000,16000,лш,1
3,good/0165e884-ccbe-4a43-b2e8-fa64952dc825.wav,3.436187,16000,рлш,1
4,good/018e28b3-b8de-4023-b2d8-39c6455c6f33.wav,5.280000,16000,стрлш,1
...,...,...,...,...,...
2777,bad/ff366031-a519-49e4-a34e-156be75f128d.wav,18.680000,16000,стлш,0
2778,bad/ff7b8da9-595b-4ee6-bd15-7016f543ec34.wav,6.501188,16000,стрлш,0
2779,bad/ffb3b065-80b4-44b2-bd89-f67e138e41d2.wav,6.760000,16000,рл,0
2780,bad/ffcc0732-1dd2-41d9-bf5d-0bbdb8f48dd0.wav,8.540000,16000,стрл,0


Сохраняем датасет для дальнейшего анализа

In [9]:
df.to_csv('dataset.csv', sep=';', index=False)